# **TAHAP 4**  Case Solution Reuse

In [ ]:
import pandas as pd

# Load Data dan buat case_solution

In [ ]:
df = pd.read_csv("/content/data/processed/cases.csv")

df["amar_putusan"] = df["amar_putusan"].fillna("").astype(str)

case_solutions = dict(
    zip(
        train_df["case_id"],
        train_df["amar_putusan"]
    )
)


# Fungsi Reuse

In [ ]:
def predict_outcome(query, k=5):

    results = retrieve_case(query, top_k=k)

    case_ids = results["case_id"].astype(str).values
    scores = results["similarity_score"].values

    weighted_solutions = {}

    for cid, score in zip(case_ids, scores):

        solution = case_solutions.get(cid, None)

        if solution is None:
            continue

        solution = str(solution).strip()

        if solution == "" or solution.lower() == "nan":
            continue

        weighted_solutions[solution] = weighted_solutions.get(solution, 0) + score

    if weighted_solutions:
        predicted_solution = max(weighted_solutions, key=weighted_solutions.get)
    else:
        predicted_solution = "No solution found"

    return {
        "predicted_solution": predicted_solution,
        "top_case_ids": list(case_ids)
    }

# Test Case

In [ ]:
query = """
pemerasan disertai pengancaman
terdakwa meminta uang dengan ancaman
"""

result = predict_outcome(query, k=5)

print("TOP CASE IDS:")
print(result["top_case_ids"])

print("\nPREDICTED SOLUTION:\n")
print(result["predicted_solution"][:800])

TOP CASE IDS:
['case_027', 'case_017', 'case_014', 'case_010', 'case_021']

PREDICTED SOLUTION:

ini; memperhatikan pasal 368ayat (1) kuhp, undang-undang nomor08 tahun 1981 tentang hukum acara pidana serta peraturan perundang-undangan lain yang bersangkutan; m e n g a d i l i 1. menyatakan terdakwa saleh tersebut di atas terbukti secara sah dan meyakinkan bersalah melakukan tindak pidana pemerasan ; 2. menjatuhkan pidana terhadapterdakwa saleh dengan pidana penjara selama 1 (satu) tahun; 3. menetapkan masa penangkapan dan penahanan yang telah dijalani terdakwadikurangkan seluruhnya dari pidana yang dijatuhkan; 4. menetapkan terdakwa tetap berada dalam tahanan; 5. menetapkan barang bukti berupa : - 1 (satu) bendel karcis jasa parkir truck forum bertais rembuk mataram yang belum terpakai; - 1 (satu) lembar karcis jasa parkir truck forum bertais rembuk mataram yang sudah terpakai no.0109


# Simpan Hasil

In [ ]:
test_queries = [
"pemerasan",
"pengancaman",
"pemerasan disertai ancaman",
"permintaan uang dengan ancaman",
"pemerasan dan pengancaman"

]

results = []

for i, q in enumerate(test_queries):

    pred = predict_outcome(q, k=5)

    results.append({
        "query_id": i + 1,
        "query": q,
        "predicted_solution": pred["predicted_solution"],
        "top_5_case_ids": ",".join(pred["top_case_ids"])
    })

# buat dataframe hasil
df_result = pd.DataFrame(results)

print(df_result)

   query_id                           query  \
0         1                       pemerasan   
1         2                     pengancaman   
2         3      pemerasan disertai ancaman   
3         4  permintaan uang dengan ancaman   
4         5       pemerasan dan pengancaman   

                                  predicted_solution  \
0  ini; memperhatikan pasal 368ayat (1) kuhp, und...   
1  ini; mengingat dan memperhatikan ketentuan per...   
2  ini; memperhatikan pasal 368ayat (1) kuhp, und...   
3  ini; memperhatikan pasal 368ayat (1) kuhp, und...   
4  ini; mengingat dan memperhatikan ketentuan per...   

                                 top_5_case_ids  
0  case_017,case_010,case_021,case_034,case_007  
1  case_014,case_025,case_017,case_010,case_030  
2  case_027,case_017,case_034,case_021,case_010  
3  case_017,case_008,case_012,case_010,case_021  
4  case_014,case_034,case_017,case_021,case_010  


In [ ]:
import os

os.makedirs(
    "/content/data/results",
    exist_ok=True
)

output_path = "/content/data/results/predictions.csv"

df_result.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nSAVED:", output_path)


SAVED: /content/data/results/predictions.csv
